In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
from math import ceil
import re
import pandas as pd

In [ ]:
exp_id = "TB-69"
data_dir = Path("Q:") / "Neutron Data" / "2-Converted_Data" / exp_id / "processed_data" / "unfiltered"
psd_dir = data_dir / "psd"
signals_dir = data_dir / "signals"

In [ ]:
psd_files = [file for file in psd_dir.iterdir() if file.suffix.lower() == ".parquet"]
signals_files = [file for file in signals_dir.iterdir() if file.suffix.lower() == ".parquet"]

In [ ]:
psd_lfs = [pl.scan_parquet(file) for file in psd_files]
psd_lf = pl.concat(psd_lfs)
signals_lfs = [pl.scan_parquet(file) for file in signals_files]
signals_lf = pl.concat(signals_lfs).cast(pl.Float32)

In [ ]:
ph_schema = {"PULSE_HEIGHT": pl.Float32}


def get_pulse_heights(
    raw_signals_df: pl.DataFrame,
    baseline_idx_range: int = 30,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pl.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()

    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)

    signals_np = -signals_np + baselines + offset
    pulse_heights = np.max(signals_np, axis=1)

    corrected_signals = pl.DataFrame(
        pulse_heights,
        ph_schema
    )
    return corrected_signals


ph_lf = signals_lf.map_batches(
    get_pulse_heights, projection_pushdown=False, schema=ph_schema
).cast(pl.String)
psd_ph_lf = pl.concat([psd_lf, ph_lf], how="horizontal")

In [ ]:
psd_ph_lf.head().collect()

In [ ]:
PATTERN = r"(.+)_(?:\d+)\.parquet"

psd_backup_dir = psd_dir.parent / "psd_backup"
signals_backup_dir = signals_dir.parent / "signals_backup"
psd_save_dir = psd_dir.parent / "psd_new"
signals_save_dir = signals_dir.parent / "signals_new"

psd_filename = psd_files[0].name
psd_filename_base = re.match(PATTERN, psd_filename).group(1)
signals_filename = signals_files[0].name
signals_filename_base = re.match(PATTERN, signals_filename).group(1)


def psd_file_path_fn(ctx: pl.BasePartitionContext) -> Path:
    return f"{psd_filename_base}_{ctx.file_idx}.parquet"


def signals_file_path_fn(ctx: pl.BasePartitionContext) -> Path:
    return f"{signals_filename_base}_{ctx.file_idx}.parquet"

In [ ]:
MAX_ROWS_PER_PART = 500000

sink_psd = psd_ph_lf.sink_parquet(
    pl.PartitionMaxSize(
        psd_save_dir,
        file_path=psd_file_path_fn,
        max_size=MAX_ROWS_PER_PART,
    ),
    mkdir=True,
    lazy=True
)
sink_signals = signals_lf.sink_parquet(
    pl.PartitionMaxSize(
        signals_save_dir,
        file_path=signals_file_path_fn,
        max_size=MAX_ROWS_PER_PART
    ),
    mkdir=True,
    lazy=True
)
print(pl.explain_all([sink_psd, sink_signals]))
pl.collect_all([sink_psd, sink_signals])
print("Done saving!")

In [ ]:
# new_psd_files = [file for file in psd_save_dir.iterdir() if file.suffix.lower() == ".parquet"]
# new_psd_lfs = [pl.scan_parquet(file) for file in new_psd_files]
# new_psd_lf = pl.concat(new_psd_lfs)
# new_psd_lf.collect_schema()

In [ ]:
# col_names = ["CALIB_ENERGY", "ENERGYSHORT", "ENERGY", "TIMETAG", "PULSE_HEIGHT"]
# df = pd.read_parquet(psd_save_dir, columns=col_names)
# df.columns